# 02 Gold Feature Coverage

This notebook validates the Gold feature layer after `src/gold/build_gold.py`
has generated the artifacts. It does not compute features directly. Its job is
to confirm grain consistency, feature coverage, and known sparse regions.

In [1]:
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

NOTEBOOK_NAME = "02_gold_feature_coverage"
GOLD = ROOT / "data" / "gold"
FEATURES = GOLD / "features"
LABELS = GOLD / "labels"
TRAINING = GOLD / "training"
METADATA = GOLD / "metadata"

OUTPUT_TABLES = ROOT / "eda" / "gold" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "gold" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "gold" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "gold" / "insights"
CHECKPOINTS = ROOT / "eda" / "gold" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

def save_chart(fig, name: str) -> None:
    fig.write_html(OUTPUT_CHARTS / f"{name}.html", include_plotlyjs="cdn")

print("=" * 72)
print(f"GOLD EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Gold root: {GOLD}")


GOLD EDA - 02_gold_feature_coverage
Start time: 2026-06-02 14:28:31.489278
Gold root: D:\F1_WinRate_Predictor\data\gold


## Feature Table Row Coverage

Every lap-grain feature table should align with the canonical master grain:
`session_key + driver_number + lap_number`. Mismatched row counts create
silent train-time nulls or dropped rows.

In [2]:
feature_files = sorted(FEATURES.glob("*.parquet"))
coverage_rows = []
for path in feature_files:
    df = pd.read_parquet(path)
    key_dupes = int(df.duplicated(["session_key", "driver_number", "lap_number"]).sum()) if {"session_key", "driver_number", "lap_number"}.issubset(df.columns) else None
    coverage_rows.append({
        "artifact": path.name,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "duplicate_keys": key_dupes,
    })
coverage = pd.DataFrame(coverage_rows).sort_values("artifact")
coverage["status"] = np.where(coverage["duplicate_keys"].fillna(0).eq(0), "PASS", "FAIL")
coverage.to_csv(OUTPUT_TABLES / "feature_row_coverage.csv", index=False)

fig = px.bar(
    coverage,
    x="artifact",
    y="rows",
    color="status",
    text="rows",
    title="Gold Feature Row Coverage by Artifact",
    labels={"artifact": "Feature artifact", "rows": "Rows"},
)
fig.update_layout(margin=dict(l=10, r=10, t=55, b=130))
fig.update_traces(texttemplate="%{text:,}", textposition="outside", cliponaxis=False)
save_chart(fig, "feature_row_coverage")
fig.show()

display(coverage)

,artifact,rows,columns,duplicate_keys,status
0,interval_lap_features.parquet,63676,9,0,PASS
1,master_lap_features.parquet,63676,58,0,PASS
2,overtake_lap_features.parquet,63676,8,0,PASS
3,pit_lap_features.parquet,63676,8,0,PASS
4,position_lap_features.parquet,63676,10,0,PASS
5,stint_lap_features.parquet,63676,10,0,PASS
6,weather_lap_features.parquet,63676,10,0,PASS


## Missingness Map

Nulls in Gold are not automatically defects. Some fields are expected to be
sparse because telemetry/event sources are sampled differently, and stint data
can be incomplete for specific sessions. The audit separates coverage risk from
pipeline failure.

In [3]:
master = pd.read_parquet(FEATURES / "master_lap_features.parquet")
null_summary = (
    master.isna().sum()
    .rename("null_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
null_summary["null_pct"] = (null_summary["null_count"] / len(master) * 100).round(3)
null_summary = null_summary.sort_values("null_pct", ascending=False)
null_summary.to_csv(OUTPUT_TABLES / "master_feature_null_profile.csv", index=False)

plot_nulls = null_summary[null_summary["null_count"].gt(0)].head(25)
fig = px.bar(
    plot_nulls,
    x="null_pct",
    y="column",
    orientation="h",
    text="null_pct",
    title="Gold Master Feature Missingness: Top Sparse Columns",
    labels={"null_pct": "Null percentage", "column": "Feature"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, margin=dict(l=10, r=10, t=55, b=10))
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside", cliponaxis=False)
save_chart(fig, "master_feature_missingness")
fig.show()

display(null_summary.head(25))

,column,null_count,null_pct
31,last_stop_duration,41694,65.478
30,last_lane_duration,25870,40.628
32,last_pit_duration,25870,40.628
29,laps_since_last_pit,25206,39.585
43,prev_gap_to_leader_seconds,8414,13.214
41,gap_to_leader_seconds,7378,11.587
42,prev_interval_value_seconds,3524,5.534
40,interval_value_seconds,2220,3.486
23,position_delta_prev_lap,1332,2.092
22,prev_position,1332,2.092


In [4]:
write_report("feature_coverage", {
    "master_rows": int(len(master)),
    "master_columns": int(len(master.columns)),
    "feature_artifacts": coverage.to_dict(orient="records"),
    "top_null_columns": null_summary.head(25).to_dict(orient="records"),
})
write_insight(
    "Gold Feature Coverage",
    [
        f"Master lap features contain {len(master):,} rows and {len(master.columns):,} columns.",
        "All generated lap-grain feature artifacts preserve canonical driver-session-lap grain.",
        "Sparse stint rows are retained rather than dropped, protecting label alignment.",
    ],
    [] if coverage["status"].eq("PASS").all() else ["At least one feature artifact has duplicate grain keys."],
    [
        "Treat high-null feature columns explicitly during model preprocessing.",
        "Investigate missing stint coverage before using tyre features as mandatory model inputs.",
    ],
)
(CHECKPOINTS / "gold_feature_coverage_completed.txt").write_text(datetime.now().isoformat(), encoding="utf-8")

26